<a href="https://colab.research.google.com/github/manurudeepika02-del/Infosys_FreightQuote_AI/blob/main/FreightQuote_AI_RAG_MyProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ FreightQuote AI — RAG Knowledge Center (Milestone 3)
### Retrieval-Augmented Generation module for the FreightQuote AI Enterprise Platform

This notebook is built to plug directly into **your** FreightQuote AI project (Milestone 1 + 2 combined
notebook). It reuses the same secrets pattern, the same `STORAGE_DIR` on Google Drive, the same
Navy & Gold UI theme, and the same Qwen-2.5-3B-Instruct (4-bit) model already used by `llm_engine.py`,
so the RAG Knowledge Center feels like one product instead of a bolted-on demo.

### What this notebook does
- **Knowledge Base Generation** — auto-builds 50+ logistics SOP / compliance / tariff PDF documents
  (customs, route optimization, carrier safety, pricing & surcharges, cargo insurance) that match the
  domain of your platform, plus 2 intentionally corrupted PDFs to prove the ingestion pipeline is
  error-resilient.
- **Ingestion Pipeline** — loads PDFs, skips corrupted ones with a logged reason, and chunks text with
  `RecursiveCharacterTextSplitter`.
- **Semantic Indexing** — embeds chunks with `all-MiniLM-L6-v2` and builds a **persistent FAISS index**
  saved to your project's `STORAGE_DIR` on Google Drive (survives Colab restarts, like your ML models).
- **LLM Answering** — reuses `Qwen/Qwen2.5-3B-Instruct` (4-bit NF4) exactly like `llm_engine.py`, with a
  high-quality CPU rule-based fallback if no GPU is available.
- **Automated Evaluation Suite** — runs 32 logistics questions end-to-end and reports pass rate + latency.
- **Streamlit Dashboard** — a Navy & Gold themed "📖 Knowledge Center" page styled to match your
  Infosys Freight Quote Portal, launched via ngrok exactly like Milestone 1 & 2.

### Before you run
In Colab → **Secrets** (key icon), make sure these exist (same names your Milestone 2 notebook uses):
`NGROK_AUTHTOKEN`, `HF_TOKEN` (optional, only needed for gated models). No other secrets are required
for this notebook.


## 📦 Step 1 — Install RAG Dependencies

In [1]:
!pip install -q reportlab pypdf langchain-text-splitters sentence-transformers faiss-cpu transformers bitsandbytes accelerate torch streamlit pyngrok streamlit-option-menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 57.9 MB/s eta 0:00:00


## 🔐 Step 2 — Load Secrets & Mount Google Drive (matches Milestone 2 `STORAGE_DIR`)

This mirrors the secrets/storage bootstrap in your Milestone 2 notebook so the FAISS index and
generated PDFs persist in the same `FreightQuote_AI` folder on Drive as your trained ML models.

In [2]:
import os

def _get_secret(key):
    # Read from Colab Secrets first, then environment variable.
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")

try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FreightQuote_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")

KNOWLEDGE_BASE_DIR = os.path.join(STORAGE_DIR, "rag", "KnowledgeBase")
INDEX_DIR          = os.path.join(STORAGE_DIR, "rag", "faiss_index")
os.makedirs(KNOWLEDGE_BASE_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

print(f"📁 Storage:        {STORAGE_DIR}")
print(f"📁 Knowledge Base: {KNOWLEDGE_BASE_DIR}")
print(f"📁 FAISS Index:    {INDEX_DIR}")
print(f"🔑 ngrok:          {'✅' if NGROK_AUTHTOKEN else '❌ set NGROK_AUTHTOKEN in Colab Secrets'}")
print(f"🔑 HF_TOKEN:       {'✅' if HF_TOKEN else '⚠️  not set (fine — Qwen2.5-3B-Instruct is ungated)'}")

Mounted at /content/drive
✅ Google Drive mounted.
📁 Storage:        /content/drive/MyDrive/FreightQuote_AI
📁 Knowledge Base: /content/drive/MyDrive/FreightQuote_AI/rag/KnowledgeBase
📁 FAISS Index:    /content/drive/MyDrive/FreightQuote_AI/rag/faiss_index
🔑 ngrok:          ❌ set NGROK_AUTHTOKEN in Colab Secrets
🔑 HF_TOKEN:       ⚠️  not set (fine — Qwen2.5-3B-Instruct is ungated)


## 🗂️ Step 3 — Write the RAG Engine (`rag_pipeline.py`)

Handles mock logistics-SOP PDF generation, error-resilient parsing, recursive text chunking,
persistent FAISS indexing, and LLM answering (Qwen-2.5-3B 4-bit, with a CPU fallback). All paths
default to the Drive-backed `KNOWLEDGE_BASE_DIR` / `INDEX_DIR` from Step 2.

In [3]:
%%writefile rag_pipeline.py
"""
rag_pipeline.py — FreightQuote AI RAG Knowledge Center
Ingestion + FAISS vector store + Qwen-2.5-3B-Instruct (4-bit) answering engine,
built to match the domain (customs, routing, carriers, pricing, insurance) and
storage conventions (Google Drive STORAGE_DIR) of the main FreightQuote AI platform.
"""
import os
import time
import json
import random
import shutil
from typing import List, Dict, Tuple, Any, Optional

import torch

# reportlab for generating mock PDFs
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# pypdf for loading PDFs
from pypdf import PdfReader

# langchain splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# sentence-transformers
from sentence_transformers import SentenceTransformer

# FAISS
import faiss
import numpy as np

# HuggingFace for LLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

random.seed(42)

# ── Storage paths (fall back to local folders if config.py / __main__ isn't available) ──
try:
    from config import STORAGE_DIR, HF_TOKEN
except Exception:
    try:
        from __main__ import STORAGE_DIR, HF_TOKEN
    except Exception:
        STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

KNOWLEDGE_BASE_DIR = os.path.join(STORAGE_DIR, "rag", "KnowledgeBase")
INDEX_DIR          = os.path.join(STORAGE_DIR, "rag", "faiss_index")
os.makedirs(KNOWLEDGE_BASE_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# ----------------- KNOWLEDGE BASE GENERATOR -----------------

def create_mock_pdf(filepath: str, title: str, paragraphs: List[str]):
    """Generates a multi-page PDF with line-wrapped paragraphs using reportlab."""
    c = canvas.Canvas(filepath, pagesize=letter)
    width, height = letter

    c.setFont("Helvetica-Bold", 16)
    c.drawString(54, height - 54, title)

    y = height - 90
    c.setFont("Helvetica", 10)

    for p_idx, para in enumerate(paragraphs):
        c.setFont("Helvetica-Bold", 11)
        c.drawString(54, y, f"Section {p_idx + 1}")
        y -= 15
        c.setFont("Helvetica", 10)

        words = para.split()
        line = ""
        for word in words:
            if c.stringWidth(line + " " + word, "Helvetica", 10) < (width - 108):
                line += " " + word
            else:
                c.drawString(54, y, line.strip())
                y -= 14
                line = word
                if y < 54:
                    c.showPage()
                    c.setFont("Helvetica", 10)
                    y = height - 54
        if line:
            c.drawString(54, y, line.strip())
            y -= 25

        if y < 80:
            c.showPage()
            c.setFont("Helvetica", 10)
            y = height - 54

    c.save()


def generate_knowledge_base(output_dir: str = KNOWLEDGE_BASE_DIR, count: int = 50):
    """Generates 50+ unique FreightQuote AI logistics SOP PDFs + 2 corrupted files."""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    topics = [
        ("Customs_Compliance_Guide", [
            "Customs clearance requires submission of Form US-382 and commercial invoices detailing HTS code classifications. Incorrect codes lead to audit penalties.",
            "International shipping transit zones require custom clearance manifests filed 48 hours prior to port arrival. Late declarations carry a tariff penalty of $500 per day.",
            "Bonded warehouses provide storage under customs authority. All cargo stored in Zone-B must satisfy security requirements under Customs Rule 409.",
            "Hazardous materials (HAZMAT) require class-specific documentation and packaging compliance. Violations lead to immediate suspension of shipping licenses.",
        ]),
        ("Route_Optimization_Protocol", [
            "Route efficiency is computed using weather parameters, port congestion levels, and berth capacity metrics. Zone 3 ocean corridors carry a default delay of 45 minutes during peak congestion.",
            "Monsoon and cyclone routing strategies dictate diversion paths when wind speed exceeds 30 knots. Alternate corridor via Suez Canal Hub is designated for heavy displacement vessels.",
            "Port congestion at Mumbai JNPT and Chennai Port requires carrier scheduling windows to avoid delays. Standard dwell time of containers averages 4.2 days.",
            "Last mile delivery optimization targets regional depots. Shipments exceeding 12,000 lbs are restricted from urban residential deliveries under city logistics laws.",
        ]),
        ("Carrier_Safety_Standard", [
            "Carrier compliance mandates tariff and safety audits every 6 months. Minimum passing tariff compliance score is 85 percent, beneath which carriers face suspension and audit flagging.",
            "Insurance liabilities require minimum coverage of 2 million dollars for active ocean and air freight carriers. Verified compliance documents must be renewed yearly.",
            "Carrier punctuality metrics log on-time delivery rate under the carrier tier rating system. Violations of punctuality thresholds carry a tier downgrade of one level.",
            "Fleet maintenance logs must verify vessel and aircraft inspection status. Non-compliant carriers are auto-flagged in the Carrier Audit dashboard.",
        ]),
        ("Pricing_and_Surcharges", [
            "Fuel surcharges are indexed weekly against the national bunker/jet fuel average. Base freight rate triggers surcharge additions when fuel index exceeds the quarterly benchmark.",
            "Peak season surcharges apply from October 1 to December 24, adding 15 percent to ocean and air freight line-haul rates across global trade routes.",
            "Less-than-Container-Load (LCL) pricing uses freight class density bands. Freight classes are determined by density, stowability, handling, and liability risk.",
            "Accessorial fees include port handling charges, detention charges ($75 per hour after 48 hours free time), inside delivery, and residential pickup surcharges.",
        ]),
        ("Logistics_Insurance_Policy", [
            "Cargo insurance coverage is limited to $100,000 standard liability unless declared value additions are requested during initial quote booking confirmation.",
            "Claims for damaged freight must be submitted within 9 days of delivery receipt, accompanied by photographic evidence and bill of lading annotations.",
            "Act of God clauses exclude coverage during extreme category 4+ cyclone or typhoon events. Alternative routing safety procedures must be documented in the risk summary.",
            "Reefer cargo temperature deviations exceeding 4 degrees Fahrenheit for more than 2 hours void standard compliance safety profiles.",
        ]),
    ]

    for idx in range(1, count + 1):
        topic_name, paragraphs = topics[(idx - 1) % len(topics)]
        unique_paras = [
            p + f" Document reference code: FREIGHT-ID-{idx:03d}-{random.randint(1000, 9999)}."
            for p in paragraphs
        ]
        filename = f"{topic_name}_Part{idx}.pdf"
        filepath = os.path.join(output_dir, filename)
        create_mock_pdf(filepath, f"FreightQuote AI SOP — Ref {idx:03d}", unique_paras)

    # 2 intentionally corrupted PDFs to prove the ingestion pipeline is error-resilient
    with open(os.path.join(output_dir, "Corrupt_Customs_Declaration.pdf"), "w") as f:
        f.write("This is a corrupted text file pretending to be a PDF header.")

    with open(os.path.join(output_dir, "Blank_Document_Error.pdf"), "wb") as f:
        f.write(b"%PDF-1.4\n%EOF")

    print(f"Generated {count} clean logistics PDFs and 2 corrupted files in folder: {output_dir}")


# ----------------- INGESTION PIPELINE -----------------

class RAGIngestionPipeline:
    def __init__(self, chunk_size: int = 600, chunk_overlap: int = 60):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
        )

    def load_and_chunk_documents(self, folder: str = KNOWLEDGE_BASE_DIR) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
        """Scans folder, loads non-corrupted PDFs, chunks text, and returns document metrics."""
        start_time = time.time()
        documents = []
        pdf_count = 0
        corrupted_count = 0
        total_pages = 0

        pdf_files = [f for f in os.listdir(folder) if f.lower().endswith(".pdf")]

        for file in pdf_files:
            filepath = os.path.join(folder, file)
            try:
                reader = PdfReader(filepath)
                pages = reader.pages
                page_count = len(pages)
                if page_count == 0:
                    raise ValueError("Empty PDF")

                text_content = ""
                for page in pages:
                    text_content += (page.extract_text() or "") + "\n"

                if not text_content.strip():
                    raise ValueError("No extractable text content")

                pdf_count += 1
                total_pages += page_count

                chunks = self.text_splitter.split_text(text_content)
                for chunk_idx, chunk in enumerate(chunks):
                    documents.append({
                        "content": chunk,
                        "metadata": {
                            "source": file,
                            "page": (chunk_idx // 2) + 1,
                            "doc_index": pdf_count,
                        },
                    })
            except Exception as e:
                corrupted_count += 1
                print(f"[LOAD ERROR] Skipping corrupted PDF '{file}': {e}")

        metrics = {
            "total_pdfs": pdf_count,
            "corrupted_pdfs": corrupted_count,
            "total_pages": total_pages,
            "total_chunks": len(documents),
            "ingestion_latency_seconds": time.time() - start_time,
        }
        return documents, metrics


# ----------------- VECTOR STORE -----------------

class PersistentRAGVectorStore:
    """FAISS vector store that also persists chunk text (not just embeddings) so it can be
    reloaded standalone by the Streamlit app without re-ingesting the PDFs."""

    def __init__(self, embedding_model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(embedding_model_name)
        self.index = None
        self.doc_metadata: List[Dict[str, Any]] = []

    def build_index(self, documents: List[Dict[str, Any]]) -> Dict[str, Any]:
        start_time = time.time()
        texts = [doc["content"] for doc in documents]

        embeddings = self.model.encode(texts, show_progress_bar=False)
        embedding_time = time.time() - start_time

        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)

        self.doc_metadata = []
        for doc in documents:
            meta = doc["metadata"].copy()
            meta["content"] = doc["content"]
            self.doc_metadata.append(meta)

        db_time = time.time() - (start_time + embedding_time)
        return {
            "embedding_time_seconds": embedding_time,
            "vector_db_creation_time_seconds": db_time,
            "total_indexed_chunks": self.index.ntotal,
        }

    def save_index(self, folder: str = INDEX_DIR):
        os.makedirs(folder, exist_ok=True)
        faiss.write_index(self.index, os.path.join(folder, "index.faiss"))
        with open(os.path.join(folder, "metadata.json"), "w") as f:
            json.dump(self.doc_metadata, f, indent=4)

    def load_index(self, folder: str = INDEX_DIR) -> bool:
        index_path = os.path.join(folder, "index.faiss")
        meta_path = os.path.join(folder, "metadata.json")
        if not (os.path.exists(index_path) and os.path.exists(meta_path)):
            return False
        self.index = faiss.read_index(index_path)
        with open(meta_path, "r") as f:
            self.doc_metadata = json.load(f)
        return True

    def search(self, query: str, top_k: int = 4) -> List[Dict[str, Any]]:
        if self.index is None or self.index.ntotal == 0:
            return []
        query_vector = self.model.encode([query])
        faiss.normalize_L2(query_vector)
        distances, indices = self.index.search(query_vector, top_k)

        results = []
        for score, idx in zip(distances[0], indices[0]):
            if idx < 0 or idx >= len(self.doc_metadata):
                continue
            results.append({"score": float(score), "metadata": self.doc_metadata[idx]})
        return results


# ----------------- LLM PIPELINE -----------------

class RAGLLM:
    """Answering engine — reuses Qwen-2.5-3B-Instruct (4-bit NF4), the same model
    llm_engine.py loads for the multi-agent dashboard, with a CPU rule-based fallback."""

    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.gpu_available = torch.cuda.is_available()
        self.memory: List[Dict[str, str]] = []

    def load_llm(self) -> bool:
        if not self.gpu_available:
            print("[RAGLLM] GPU not detected. Using CPU rule-based generation fallback.")
            return False

        try:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            kw = {"token": HF_TOKEN} if HF_TOKEN else {}
            self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
            self.model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, quantization_config=bnb_config, device_map="auto", **kw
            )
            print("[RAGLLM] Quantized 4-bit Qwen-2.5-3B loaded successfully on GPU.")
            return True
        except Exception as e:
            print(f"[RAGLLM] GPU loading failed: {e}. Falling back to CPU mode.")
            self.model, self.tokenizer = None, None
            return False

    def generate_answer(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Tuple[str, float]:
        start_time = time.time()

        context_str = ""
        for idx, chunk in enumerate(retrieved_chunks):
            context_str += (
                f"[Source {idx + 1}: {chunk['metadata']['source']} "
                f"(Page {chunk['metadata']['page']})]\n{chunk['metadata']['content']}\n\n"
            )

        history_str = ""
        for turn in self.memory[-3:]:
            history_str += f"User: {turn['query']}\nAI: {turn['answer']}\n"

        system_prompt = (
            "You are the FreightQuote AI Knowledge Center Assistant. Answer the user's question "
            "using ONLY the provided document contexts. If the answer cannot be found in the "
            "context, say: 'I cannot find the answer in the provided documents.' "
            "Do not make up facts or use external knowledge."
        )

        prompt = f"""Document Contexts:
{context_str}

Conversation History:
{history_str}

Question: {query}

Provide a concise, direct answer based strictly on the contexts. Cite the Source files and Page numbers in your response."""

        latency = 0.0
        answer = ""

        if self.model is not None and self.tokenizer is not None:
            try:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ]
                text = self.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

                with torch.inference_mode():
                    generated_ids = self.model.generate(
                        **model_inputs, max_new_tokens=256, do_sample=False, use_cache=True
                    )
                generated_ids = [
                    output_ids[len(input_ids):]
                    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
                ]
                answer = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
                latency = time.time() - start_time
            except Exception as e:
                print(f"[RAGLLM] GPU inference failed: {e}. Executing CPU fallback generator.")
                self.model = None

        if self.model is None:
            answer = self.fallback_rule_based_qa(query, retrieved_chunks)
            latency = time.time() - start_time

        self.memory.append({"query": query, "answer": answer})
        return answer, latency

    def fallback_rule_based_qa(self, query: str, chunks: List[Dict[str, Any]]) -> str:
        """Heuristic keyword-overlap QA engine used when no GPU is available."""
        if not chunks:
            return "I cannot find the answer in the provided documents."

        query_words = [w.lower() for w in query.replace("?", "").split() if len(w) > 3]
        best_chunk = chunks[0]
        max_overlap = 0

        for chunk in chunks:
            content_lower = chunk["metadata"]["content"].lower()
            overlap = sum(1 for w in query_words if w in content_lower)
            if overlap > max_overlap:
                max_overlap = overlap
                best_chunk = chunk

        source = best_chunk["metadata"]["source"]
        page = best_chunk["metadata"]["page"]
        content = best_chunk["metadata"]["content"]

        sentences = content.split(".")
        relevant = [s.strip() for s in sentences if any(w in s.lower() for w in query_words)]
        extracted = ". ".join(relevant[:2]) + "." if relevant else content.strip().split("\n")[0]

        return f"{extracted} [Source: {source}, Page: {page}]"


Writing rag_pipeline.py


## 🧪 Step 4 — Write the Automated Evaluation Engine (`evaluator.py`)

32 targeted logistics questions covering customs, routing, carrier safety, pricing, and cargo
insurance — the same categories FreightQuote AI's Agents 1–3 already reason about.

In [4]:
%%writefile evaluator.py
"""evaluator.py — automated 32-question evaluation suite for the FreightQuote AI RAG engine."""
import time
from typing import List, Dict, Any, Tuple

import pandas as pd

from rag_pipeline import PersistentRAGVectorStore, RAGLLM


def get_eval_questions() -> List[str]:
    """Returns 32 logistics questions covering all core FreightQuote AI knowledge categories."""
    return [
        # Customs Compliance (1-6)
        "What forms are required for customs clearance?",
        "What is the penalty for late custom clearance declarations?",
        "Under what rule must Zone-B cargo satisfy customs security?",
        "What happens if HAZMAT packaging violates compliance regulations?",
        "What is the filing window for custom clearance manifests?",
        "HTS code classification is required on what documents?",

        # Route Optimization (7-12)
        "How is routing efficiency computed?",
        "What corridor carries a 45 minute peak congestion delay?",
        "Which corridor is the alternate route for heavy displacement vessels?",
        "What is the average dwell time at Mumbai JNPT and Chennai Port?",
        "What shipment weight is restricted from urban residential delivery?",
        "When do monsoon or cyclone conditions trigger routing diversions?",

        # Carrier Safety & Compliance (13-18)
        "How often must carrier safety and tariff audits be conducted?",
        "What is the minimum passing tariff compliance score?",
        "What is the required insurance coverage limit for active carriers?",
        "What happens to a carrier's tier rating after a punctuality violation?",
        "What triggers an automatic flag in the Carrier Audit dashboard?",
        "What logs must verify carrier fleet maintenance status?",

        # Pricing & Surcharges (19-25)
        "How are fuel surcharges indexed?",
        "When do peak season surcharges apply?",
        "What parameters determine LCL freight pricing?",
        "What is the hourly rate for carrier detention after free time?",
        "When does the fuel index trigger additional surcharges?",
        "What accessorial fees are charged by carriers?",
        "What percentage is added for peak season line-haul rates?",

        # Cargo Insurance (26-32)
        "What is the standard liability insurance limit for cargo?",
        "Within how many days must freight damage claims be submitted?",
        "Are category 4 cyclone events covered under standard clauses?",
        "What reefer temperature deviations void compliance safety?",
        "What documents must accompany photographic evidence for cargo claims?",
        "Does standard insurance cover Acts of God?",
        "How many hours of temperature deviation will void reefer compliance?",
    ]


def run_evaluation(vector_store: PersistentRAGVectorStore, llm: RAGLLM) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Runs the 32-query evaluation suite, tracks latency and retrieval accuracy, and compiles a summary."""
    questions = get_eval_questions()
    results = []

    total_latency = 0.0
    total_retrieval_latency = 0.0
    passed_queries = 0

    print(f"Starting automated evaluation of {len(questions)} logistics queries...")

    for idx, query in enumerate(questions):
        retrieval_start = time.time()
        hits = vector_store.search(query, top_k=3)
        total_retrieval_latency += time.time() - retrieval_start

        ans_start = time.time()
        answer, _ = llm.generate_answer(query, hits)
        ans_latency = time.time() - ans_start
        total_latency += ans_latency

        top_score = hits[0]["score"] if hits else 0.0
        status = "Pass" if (hits and top_score > 0.25 and "cannot find the answer" not in answer.lower()) else "Fail"
        if status == "Pass":
            passed_queries += 1

        retrieved_doc = hits[0]["metadata"]["source"] if hits else "None"
        retrieved_page = hits[0]["metadata"]["page"] if hits else "None"

        results.append({
            "Query ID": f"Q-{idx + 1:02d}",
            "Question": query,
            "Retrieved Source": retrieved_doc,
            "Page": retrieved_page,
            "Similarity Score": f"{top_score:.4f}",
            "Answer": answer[:120] + ("..." if len(answer) > 120 else ""),
            "Status": status,
            "Latency (s)": f"{ans_latency:.3f}",
        })

        print(f"[{idx + 1}/{len(questions)}] Query ID: Q-{idx + 1:02d} | Status: {status} | Latency: {ans_latency:.2f}s")

    df = pd.DataFrame(results)

    summary = {
        "passed_queries": passed_queries,
        "total_queries": len(questions),
        "average_generation_latency": total_latency / len(questions),
        "average_retrieval_latency": total_retrieval_latency / len(questions),
        "pass_rate_percent": (passed_queries / len(questions)) * 100,
    }

    return df, summary


Writing evaluator.py


## 🖥️ Step 5 — Write the Streamlit Dashboard (`app.py`)

Styled with the same Navy & Gold theme as your Infosys Freight Quote Portal (Milestone 1 & 2), so it
looks like a native "📖 Knowledge Center" page of the same product rather than a separate demo.

In [5]:
%%writefile app.py
"""app.py — FreightQuote AI RAG Knowledge Center (Navy & Gold theme, matches Milestone 1 & 2 UI)"""
import os
import time

import streamlit as st

from rag_pipeline import PersistentRAGVectorStore, RAGLLM, KNOWLEDGE_BASE_DIR, INDEX_DIR

st.set_page_config(page_title="Infosys Freight Quote Portal — Knowledge Center", page_icon="📖", layout="wide")

COLORS = {
    "navy_deep": "#0b1530", "navy": "#0f1c3f", "gold": "#c9a24b", "gold_hover": "#b8912f",
    "bg_main": "#f4f5f7", "bg_card": "#ffffff", "text_heading": "#0b1530", "text_body": "#1b2436",
    "text_muted": "#5b6478", "border": "#d8dbe2", "green": "#2f8f5b", "red": "#b3413a",
}

st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700;800&family=Inter:wght@400;500;600&display=swap');
html, body, [class*="css"] {{ font-family: 'Inter', sans-serif; color: {COLORS["text_body"]}; background-color: {COLORS["bg_main"]}; }}
h1, h2, h3 {{ font-family: 'Playfair Display', serif; color: {COLORS["text_heading"]} !important; font-weight: 700; }}
.chunk-card {{
    background: {COLORS["bg_card"]}; border: 1px solid {COLORS["border"]}; border-left: 4px solid {COLORS["gold"]};
    border-radius: 10px; padding: 16px; margin-bottom: 12px; box-shadow: 0 2px 10px rgba(11,21,48,0.06);
}}
.metric-badge {{
    display: inline-block; background: {COLORS["navy"]}; color: {COLORS["gold_light"] if "gold_light" in COLORS else "#e8d9ad"};
    padding: 4px 12px; border-radius: 9999px; font-size: 12px; font-weight: 600; margin-right: 8px;
}}
.stTextInput>div>div>input {{ border: 1px solid {COLORS["border"]} !important; border-radius: 8px !important; }}
.stButton>button {{
    background: {COLORS["gold"]}; color: {COLORS["navy_deep"]}; border: none; border-radius: 8px;
    padding: 10px 24px; font-weight: 700;
}}
.stButton>button:hover {{ background: {COLORS["gold_hover"]}; }}
</style>
""", unsafe_allow_html=True)


@st.cache_resource
def load_rag_systems():
    """Initializes and caches the FAISS vector store and Qwen LLM for the Streamlit UI."""
    vector_store = PersistentRAGVectorStore()
    if os.path.exists(INDEX_DIR):
        loaded = vector_store.load_index(INDEX_DIR)
        print("[Streamlit App] Loaded persistent FAISS index." if loaded else "[Streamlit App] Index folder present but empty.")
    else:
        print("[Streamlit App] WARNING: FAISS index folder not found. Run the ingestion pipeline in the notebook first.")

    llm = RAGLLM()
    llm.load_llm()
    return vector_store, llm


vector_store, llm = load_rag_systems()

st.markdown(f'<div style="text-align:center;padding:6px 0;font-weight:700;font-size:20px;color:{COLORS["text_heading"]};">🏛️ Infosys Freight Quote Portal</div>', unsafe_allow_html=True)
st.title("📖 RAG Knowledge Center")
st.markdown(f"<p style='color:{COLORS['text_muted']};'>Semantic Search & Intelligent QA over Logistics SOPs, Customs, Carrier & Pricing Policies</p>", unsafe_allow_html=True)

st.sidebar.markdown("### 📊 System Status")
if llm.gpu_available and llm.model is not None:
    st.sidebar.success("🟢 GPU Mode: Qwen2.5-3B (4-bit)")
else:
    st.sidebar.info("🔵 Fallback Mode: CPU Rule-Based Heuristic")

if vector_store.index is not None:
    st.sidebar.markdown(f"**Index size**: `{vector_store.index.ntotal}` text chunks")

if os.path.exists(KNOWLEDGE_BASE_DIR):
    pdf_files = [f for f in os.listdir(KNOWLEDGE_BASE_DIR) if f.lower().endswith(".pdf")]
    st.sidebar.markdown(f"**Indexed documents**: `{len(pdf_files)}` PDFs")
else:
    st.sidebar.markdown("**Indexed documents**: `0` PDFs")

query = st.text_input("Ask a logistics / customs / carrier / pricing question:",
                       placeholder="e.g., What is the detention charge after free time?")

if st.button("Search Knowledge Base") or query:
    if not query.strip():
        st.warning("Please enter a question.")
    elif vector_store.index is None:
        st.error("FAISS vector database is not loaded. Please run the ingestion pipeline in the notebook first.")
    else:
        with st.spinner("Searching documents & generating answer..."):
            retrieval_start = time.time()
            hits = vector_store.search(query, top_k=4)
            retrieval_latency = time.time() - retrieval_start

            answer, gen_latency = llm.generate_answer(query, hits)

            st.markdown("### 🤖 Answer")
            st.info(answer)

            st.markdown(
                f"<span class='metric-badge'>Retrieval: {retrieval_latency:.4f}s</span>"
                f"<span class='metric-badge'>Generation: {gen_latency:.4f}s</span>"
                f"<span class='metric-badge'>Chunks searched: {len(hits)}</span>",
                unsafe_allow_html=True,
            )

            st.markdown("### 📄 Retrieved Context Chunks")
            for idx, hit in enumerate(hits):
                score = hit["score"]
                meta = hit["metadata"]
                st.markdown(f"""
                <div class="chunk-card">
                    <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
                        <span style="font-weight:bold;color:{COLORS['navy']};">[Chunk {idx + 1}] {meta['source']} (Page {meta['page']})</span>
                        <span style="color:{COLORS['green']};font-weight:bold;">Score: {score:.4f}</span>
                    </div>
                    <div style="font-size:13px;color:{COLORS['text_body']};line-height:1.5;">{meta['content']}</div>
                </div>
                """, unsafe_allow_html=True)

if st.sidebar.button("Clear Conversation Memory"):
    llm.memory = []
    st.sidebar.success("Memory cleared.")


Writing app.py


## ⚙️ Step 6 — Run Ingestion & Build the Vector Index

Generates the 50+ logistics PDFs (+2 corrupted files) into `KNOWLEDGE_BASE_DIR`, chunks them, embeds
the chunks, builds the FAISS index, and persists it to `INDEX_DIR` on Google Drive.

In [6]:
import rag_pipeline

# 1. Generate 50+ unique PDFs + 2 corrupted files
rag_pipeline.generate_knowledge_base(output_dir=rag_pipeline.KNOWLEDGE_BASE_DIR, count=50)

# 2. Ingest documents
print("\nStarting loading and chunking process...")
ingester = rag_pipeline.RAGIngestionPipeline(chunk_size=600, chunk_overlap=60)
docs, ingest_metrics = ingester.load_and_chunk_documents(folder=rag_pipeline.KNOWLEDGE_BASE_DIR)
print(f"Ingestion completed. Metrics: {ingest_metrics}")

# 3. Build the FAISS vector index
print("\nEncoding text chunks and building FAISS index...")
vector_store = rag_pipeline.PersistentRAGVectorStore()
store_metrics = vector_store.build_index(docs)
print(f"Vector DB completed. Metrics: {store_metrics}")

# 4. Persist to Google Drive
vector_store.save_index(rag_pipeline.INDEX_DIR)
print(f"✅ FAISS index persisted to: {rag_pipeline.INDEX_DIR}")

Generated 50 clean logistics PDFs and 2 corrupted files in folder: /content/drive/MyDrive/FreightQuote_AI/rag/KnowledgeBase

Starting loading and chunking process...


[LOAD ERROR] Skipping corrupted PDF 'Corrupt_Customs_Declaration.pdf': Stream has ended unexpectedly
[LOAD ERROR] Skipping corrupted PDF 'Blank_Document_Error.pdf': Stream has ended unexpectedly
Ingestion completed. Metrics: {'total_pdfs': 50, 'corrupted_pdfs': 2, 'total_pages': 50, 'total_chunks': 100, 'ingestion_latency_seconds': 0.29227161407470703}

Encoding text chunks and building FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB completed. Metrics: {'embedding_time_seconds': 0.8925411701202393, 'vector_db_creation_time_seconds': 0.0003418922424316406, 'total_indexed_chunks': 100}
✅ FAISS index persisted to: /content/drive/MyDrive/FreightQuote_AI/rag/faiss_index


## 📊 Step 7 — Run the Automated 32-Query Evaluation Suite

Loads the persisted index + Qwen-2.5-3B, runs all 32 questions, and renders a metrics report.

In [7]:
import evaluator
import rag_pipeline
from IPython.display import display, Markdown

# Load vector index
v_store = rag_pipeline.PersistentRAGVectorStore()
loaded = v_store.load_index(rag_pipeline.INDEX_DIR)
if not loaded:
    raise RuntimeError("FAISS index not found — run Step 6 first.")

# Load LLM (Qwen-2.5-3B 4-bit on GPU, or CPU fallback)
llm = rag_pipeline.RAGLLM()
llm.load_llm()

# Run evaluation suite
df_results, summary_metrics = evaluator.run_evaluation(v_store, llm)

print("\n" + "=" * 60)
print("                  EVALUATION REPORT SUMMARY")
print("=" * 60)
print(f"Indexed Chunks      : {v_store.index.ntotal}")
print(f"Passed Queries      : {summary_metrics['passed_queries']}/{summary_metrics['total_queries']}")
print(f"Average Gen Latency : {summary_metrics['average_generation_latency']:.4f} seconds")
print(f"Average Ret Latency : {summary_metrics['average_retrieval_latency']:.4f} seconds")
print(f"Pass Rate           : {summary_metrics['pass_rate_percent']:.1f}%")
print("=" * 60 + "\n")

display(Markdown("### Detailed Query Metrics"))
display(Markdown(df_results.to_markdown(index=False)))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[RAGLLM] Quantized 4-bit Qwen-2.5-3B loaded successfully on GPU.
Starting automated evaluation of 32 logistics queries...
[1/32] Query ID: Q-01 | Status: Pass | Latency: 4.32s
[2/32] Query ID: Q-02 | Status: Pass | Latency: 3.56s
[3/32] Query ID: Q-03 | Status: Pass | Latency: 3.35s
[4/32] Query ID: Q-04 | Status: Pass | Latency: 4.13s
[5/32] Query ID: Q-05 | Status: Pass | Latency: 3.34s
[6/32] Query ID: Q-06 | Status: Pass | Latency: 3.09s
[7/32] Query ID: Q-07 | Status: Pass | Latency: 3.97s
[8/32] Query ID: Q-08 | Status: Pass | Latency: 5.59s
[9/32] Query ID: Q-09 | Status: Pass | Latency: 5.56s
[10/32] Query ID: Q-10 | Status: Pass | Latency: 4.41s
[11/32] Query ID: Q-11 | Status: Fail | Latency: 1.15s
[12/32] Query ID: Q-12 | Status: Pass | Latency: 3.75s
[13/32] Query ID: Q-13 | Status: Pass | Latency: 3.04s
[14/32] Query ID: Q-14 | Status: Pass | Latency: 3.53s
[15/32] Query ID: Q-15 | Status: Pass | Latency: 3.77s
[16/32] Query ID: Q-16 | Status: Pass | Latency: 2.69s
[17/32]

### Detailed Query Metrics

| Query ID   | Question                                                               | Retrieved Source                       |   Page |   Similarity Score | Answer                                                                                                                      | Status   |   Latency (s) |
|:-----------|:-----------------------------------------------------------------------|:---------------------------------------|-------:|-------------------:|:----------------------------------------------------------------------------------------------------------------------------|:---------|--------------:|
| Q-01       | What forms are required for customs clearance?                         | Customs_Compliance_Guide_Part31.pdf    |      1 |             0.5156 | For customs clearance, the required form is Form US-382. This information can be found in all three sources (Source 1, S... | Pass     |         4.321 |
| Q-02       | What is the penalty for late custom clearance declarations?            | Customs_Compliance_Guide_Part46.pdf    |      1 |             0.5143 | The penalty for late custom clearance declarations is $500 per day. This information can be found in all three sources (... | Pass     |         3.559 |
| Q-03       | Under what rule must Zone-B cargo satisfy customs security?            | Customs_Compliance_Guide_Part46.pdf    |      1 |             0.6569 | According to Source 1, Source 2, and Source 3 on Page 1, Zone-B cargo must satisfy customs security under Rule Section 3... | Pass     |         3.352 |
| Q-04       | What happens if HAZMAT packaging violates compliance regulations?      | Customs_Compliance_Guide_Part46.pdf    |      1 |             0.7444 | If HAZMAT packaging violates compliance regulations, it leads to immediate suspension of shipping licenses. This informa... | Pass     |         4.125 |
| Q-05       | What is the filing window for custom clearance manifests?              | Customs_Compliance_Guide_Part46.pdf    |      1 |             0.3096 | The filing window for custom clearance manifests is 48 hours prior to port arrival. This information can be found in Sou... | Pass     |         3.338 |
| Q-06       | HTS code classification is required on what documents?                 | Customs_Compliance_Guide_Part1.pdf     |      1 |             0.4399 | According to Source 1, Source 2, and Source 3 on Page 1, HTS code classification is required on documents related to Haz... | Pass     |         3.094 |
| Q-07       | How is routing efficiency computed?                                    | Route_Optimization_Protocol_Part47.pdf |      1 |             0.5326 | Routing efficiency is computed using weather parameters, port congestion levels, and berth capacity metrics. This inform... | Pass     |         3.968 |
| Q-08       | What corridor carries a 45 minute peak congestion delay?               | Route_Optimization_Protocol_Part37.pdf |      1 |             0.4508 | According to Source 1, Page 1 and Source 2, Page 1, Zone 3 ocean corridors carry a 45 minute peak congestion delay.         | Pass     |         5.594 |
|            |                                                                        |                                        |        |                    |                                                                                                                             |          |               |
|            |                                                                        |                                        |        |                    | Sou...                                                                                                                      |          |               |
| Q-09       | Which corridor is the alternate route for heavy displacement vessels?  | Route_Optimization_Protocol_Part37.pdf |      1 |             0.5107 | The alternate route for heavy displacement vessels is the Suez Canal Hub. This information can be found in Source 1, Pag... | Pass     |         5.558 |
| Q-10       | What is the average dwell time at Mumbai JNPT and Chennai Port?        | Route_Optimization_Protocol_Part22.pdf |      1 |             0.6575 | The average dwell time at Mumbai JNPT and Chennai Port is 4.2 days. This information can be found in Source 1, Page 1 an... | Pass     |         4.41  |
| Q-11       | What shipment weight is restricted from urban residential delivery?    | Pricing_and_Surcharges_Part49.pdf      |      1 |             0.4606 | I cannot find the answer in the provided documents.                                                                         | Fail     |         1.15  |
| Q-12       | When do monsoon or cyclone conditions trigger routing diversions?      | Route_Optimization_Protocol_Part37.pdf |      1 |             0.5095 | When monsoon or cyclone conditions occur and wind speeds exceed 30 knots, routing diversions are triggered. This informa... | Pass     |         3.748 |
| Q-13       | How often must carrier safety and tariff audits be conducted?          | Carrier_Safety_Standard_Part43.pdf     |      1 |             0.6881 | Carrier safety and tariff audits must be conducted every 6 months. This information can be found in Source 1, Page 1 and... | Pass     |         3.045 |
| Q-14       | What is the minimum passing tariff compliance score?                   | Carrier_Safety_Standard_Part3.pdf      |      1 |             0.6528 | The minimum passing tariff compliance score is 85 percent. This information can be found in Source 1, Page 1 and Source ... | Pass     |         3.527 |
| Q-15       | What is the required insurance coverage limit for active carriers?     | Carrier_Safety_Standard_Part48.pdf     |      1 |             0.4942 | The required insurance coverage limit for active carriers is 2 million dollars. This information can be found in Source ... | Pass     |         3.774 |
| Q-16       | What happens to a carrier's tier rating after a punctuality violation? | Carrier_Safety_Standard_Part28.pdf     |      1 |             0.6753 | According to Section 3 of Source 1 and Source 2, when a carrier violates punctuality thresholds, their tier rating is do... | Pass     |         2.687 |
| Q-17       | What triggers an automatic flag in the Carrier Audit dashboard?        | Carrier_Safety_Standard_Part28.pdf     |      1 |             0.4877 | According to Section 3 of Source 1 and Source 2, fleet maintenance logs trigger an automatic flag in the Carrier Audit d... | Pass     |         2.89  |
| Q-18       | What logs must verify carrier fleet maintenance status?                | Carrier_Safety_Standard_Part13.pdf     |      1 |             0.694  | According to Section 3 of Source 1 and Source 2, fleet maintenance logs must verify carrier fleet maintenance status. Th... | Pass     |         3.468 |
| Q-19       | How are fuel surcharges indexed?                                       | Pricing_and_Surcharges_Part49.pdf      |      1 |             0.6596 | According to Section 1 of Source 1 and Source 2, fuel surcharges are indexed weekly against the national bunker/jet fuel... | Pass     |         4.683 |
| Q-20       | When do peak season surcharges apply?                                  | Pricing_and_Surcharges_Part14.pdf      |      1 |             0.5728 | According to Section 2 of Source 1 and Source 2, peak season surcharges apply from October 1 to December 24, adding 15 p... | Pass     |         5.429 |
| Q-21       | What parameters determine LCL freight pricing?                         | Pricing_and_Surcharges_Part4.pdf       |      1 |             0.6009 | According to Section 3 of Source 1 and Source 2, LCL freight pricing uses freight class density bands, which are determi... | Pass     |         4.307 |
| Q-22       | What is the hourly rate for carrier detention after free time?         | Pricing_and_Surcharges_Part44.pdf      |      1 |             0.574  | According to Section 4 of Source 1 and Source 2, the hourly rate for carrier detention after free time is $75 per hour. ... | Pass     |         7.037 |
| Q-23       | When does the fuel index trigger additional surcharges?                | Pricing_and_Surcharges_Part24.pdf      |      1 |             0.66   | According to Section 1 of Source 1 and Source 2, the fuel index triggers additional surcharges when it exceeds the quart... | Pass     |         6.097 |
| Q-24       | What accessorial fees are charged by carriers?                         | Pricing_and_Surcharges_Part44.pdf      |      1 |             0.5649 | According to Section 4 of Source 1 and Source 2, the accessorial fees charged by carriers include port handling charges,... | Pass     |         8.683 |
| Q-25       | What percentage is added for peak season line-haul rates?              | Pricing_and_Surcharges_Part49.pdf      |      1 |             0.5921 | According to Section 2 of Source 1 and Source 2, the peak season surcharges add 15 percent to ocean and air freight line... | Pass     |         7.378 |
| Q-26       | What is the standard liability insurance limit for cargo?              | Logistics_Insurance_Policy_Part45.pdf  |      1 |             0.6975 | According to Section 1 of Source 1 and Source 2, the standard liability insurance limit for cargo is $100,000. This info... | Pass     |         7.01  |
| Q-27       | Within how many days must freight damage claims be submitted?          | Logistics_Insurance_Policy_Part45.pdf  |      1 |             0.5959 | According to Section 1 of Source 1 and Source 2, freight damage claims must be submitted within 9 days of delivery recei... | Pass     |         6.533 |
| Q-28       | Are category 4 cyclone events covered under standard clauses?          | Logistics_Insurance_Policy_Part10.pdf  |      1 |             0.2719 | I cannot find the answer in the provided documents.                                                                         | Fail     |         1.412 |
| Q-29       | What reefer temperature deviations void compliance safety?             | Logistics_Insurance_Policy_Part15.pdf  |      1 |             0.6626 | According to Section 4 of Source 1 and Source 2, reefer cargo temperature deviations exceeding 4 degrees Fahrenheit for ... | Pass     |         6.96  |
| Q-30       | What documents must accompany photographic evidence for cargo claims?  | Logistics_Insurance_Policy_Part45.pdf  |      1 |             0.3378 | According to Section 1 of Source 1 and Source 2, claims for damaged freight must be submitted with photographic evidence... | Pass     |         6.9   |
| Q-31       | Does standard insurance cover Acts of God?                             | Logistics_Insurance_Policy_Part30.pdf  |      1 |             0.3865 | According to Section 1 of Source 1 and Source 2, Acts of God clauses exclude coverage during extreme category 4+ cyclone... | Pass     |         6.85  |
| Q-32       | How many hours of temperature deviation will void reefer compliance?   | Logistics_Insurance_Policy_Part50.pdf  |      1 |             0.6133 | According to Section 4 of Source 1 and Source 2, reefer cargo temperature deviations exceeding 4 degrees Fahrenheit for ... | Pass     |         7.142 |

## 🚀 Step 8 — Launch the Streamlit Knowledge Center via ngrok

Uses the same `NGROK_AUTHTOKEN` secret and launch pattern as your Milestone 1 & 2 notebook.

In [8]:
import subprocess, time
from pyngrok import ngrok

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless=true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)

if NGROK_AUTHTOKEN:
    public_url = ngrok.connect(8501, proto="http").public_url
    print("=" * 60)
    print(f"📖 RAG Knowledge Center is Live!")
    print(f"Access URL: {public_url}")
    print("=" * 60)
else:
    print("⚠️  NGROK_AUTHTOKEN not set — app is running locally on port 8501 only.")

print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Received stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")

⚠️  NGROK_AUTHTOKEN not set — app is running locally on port 8501 only.
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.

🛑 Received stop signal. Shutting down...
✅ Ngrok tunnel closed and Streamlit server stopped gracefully.


## 🛑 Step 9 — Stop the App & Free GPU Memory

In [9]:
try:
    process.terminate()
    ngrok.kill()
    print("🛑 Streamlit and ngrok terminated successfully.")
except Exception as e:
    print("Info:", e)

🛑 Streamlit and ngrok terminated successfully.


## 📄 Step 10 — Requirements & Screenshot Checklist

In [10]:
requirements = """
reportlab>=4.0.0
pypdf>=3.10.0
langchain-text-splitters>=0.0.1
sentence-transformers>=2.2.2
faiss-cpu>=1.7.4
transformers>=4.31.0
bitsandbytes>=0.41.0
accelerate>=0.21.0
torch>=2.0.0
streamlit>=1.25.0
pyngrok>=6.0.0
streamlit-option-menu>=0.3.6
"""
with open("requirements.txt", "w") as f:
    f.write(requirements.strip())
print("✅ requirements.txt written.\n")

checklist = """
===========================================================
               RAG SCREENSHOTS VERIFICATION CHECKLIST
===========================================================
[ ] 1. Ingestion console output (50 PDFs loaded, pages, chunks)
[ ] 2. Corrupt PDF skipping warning logs
[ ] 3. Embeddings + FAISS index creation duration report
[ ] 4. Evaluation: 32-query execution log
[ ] 5. Markdown results report table (renders in notebook)
[ ] 6. Average generation/retrieval latency stats
[ ] 7. Streamlit Knowledge Center search screen (Navy & Gold theme)
[ ] 8. Retrieved chunk cards with similarity score
[ ] 9. Ngrok live deployment link
"""
print(checklist)

✅ requirements.txt written.


               RAG SCREENSHOTS VERIFICATION CHECKLIST
[ ] 1. Ingestion console output (50 PDFs loaded, pages, chunks)
[ ] 2. Corrupt PDF skipping warning logs
[ ] 3. Embeddings + FAISS index creation duration report
[ ] 4. Evaluation: 32-query execution log
[ ] 5. Markdown results report table (renders in notebook)
[ ] 6. Average generation/retrieval latency stats
[ ] 7. Streamlit Knowledge Center search screen (Navy & Gold theme)
[ ] 8. Retrieved chunk cards with similarity score
[ ] 9. Ngrok live deployment link

